<a href="https://colab.research.google.com/github/Ahmadlazz/akuisisi-cuaca-bali/blob/main/Open_Meteo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
import pandas as pd
import json
import os
import hashlib

from datetime import datetime
from zoneinfo import ZoneInfo

print("Semua library berhasil diimpor!")

Semua library berhasil diimpor!


In [5]:
# Lokasi penelitian
latitude = -8.65
longitude = 115.22

lokasi = "Denpasar, Bali"

# Periode pengambilan data
tanggal_mulai = "2026-11-01"
tanggal_selesai = "2026-11-07"

# Sumber API
url = "https://archive-api.open-meteo.com/v1/archive"

# Parameter permintaan
parameter = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": tanggal_mulai,
    "end_date": tanggal_selesai,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "cloud_cover",
        "weather_code"
    ],
    "timezone": "Asia/Singapore"
}

print("Parameter pengambilan data berhasil disiapkan!")

Parameter pengambilan data berhasil disiapkan!


In [6]:
# Mengirim permintaan ke Open-Meteo
response = requests.get(
    url,
    params=parameter,
    timeout=30
)

# Memeriksa respons API
response.raise_for_status()

# Mengubah respons menjadi JSON
hasil_api = response.json()

print("Berhasil mengambil data dari Open-Meteo!")
print("Status HTTP:", response.status_code)
print("Zona waktu:", hasil_api["timezone"])

Berhasil mengambil data dari Open-Meteo!
Status HTTP: 200
Zona waktu: Asia/Singapore


In [7]:
# Mengambil bagian data per jam
data = pd.DataFrame(hasil_api["hourly"])

# Mengganti nama kolom
data = data.rename(columns={
    "time": "waktu",
    "temperature_2m": "suhu_celsius",
    "relative_humidity_2m": "kelembapan_persen",
    "precipitation": "curah_hujan_mm",
    "wind_speed_10m": "kecepatan_angin_kmh",
    "cloud_cover": "tutupan_awan_persen",
    "weather_code": "kode_cuaca"
})

# Menambahkan informasi lokasi
data["lokasi"] = lokasi

# Mengatur urutan kolom
data = data[
    [
        "waktu",
        "lokasi",
        "suhu_celsius",
        "kelembapan_persen",
        "curah_hujan_mm",
        "kecepatan_angin_kmh",
        "tutupan_awan_persen",
        "kode_cuaca"
    ]
]

print("Data berhasil diubah menjadi DataFrame!")

Data berhasil diubah menjadi DataFrame!


In [8]:
# Melihat lima data pertama
print("Lima data pertama:")
display(data.head())

# Melihat jumlah baris dan kolom
print("\nUkuran dataset:")
print(data.shape)

# Memeriksa informasi kolom
print("\nInformasi dataset:")
data.info()

# Memeriksa data kosong
print("\nJumlah data kosong:")
print(data.isnull().sum())

# Memeriksa jumlah record
print("\nTotal record:", len(data))

if len(data) >= 100:
    print("Dataset memenuhi minimal 100 record!")
else:
    print("Dataset belum memenuhi minimal 100 record!")

Lima data pertama:


,waktu,lokasi,suhu_celsius,kelembapan_persen,curah_hujan_mm,kecepatan_angin_kmh,tutupan_awan_persen,kode_cuaca
0,2026-11-01T00:00,"Denpasar, Bali",24.2,95,0.0,10.3,99,3
1,2026-11-01T01:00,"Denpasar, Bali",24.2,93,0.0,8.3,99,3
2,2026-11-01T02:00,"Denpasar, Bali",24.4,93,0.0,5.1,100,3
3,2026-11-01T03:00,"Denpasar, Bali",24.3,94,0.0,3.8,100,3
4,2026-11-01T04:00,"Denpasar, Bali",24.3,93,0.0,3.8,97,3



Ukuran dataset:
(168, 8)

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   waktu                168 non-null    object 
 1   lokasi               168 non-null    object 
 2   suhu_celsius         168 non-null    float64
 3   kelembapan_persen    168 non-null    int64  
 4   curah_hujan_mm       168 non-null    float64
 5   kecepatan_angin_kmh  168 non-null    float64
 6   tutupan_awan_persen  168 non-null    int64  
 7   kode_cuaca           168 non-null    int64  
dtypes: float64(3), int64(3), object(2)
memory usage: 10.6+ KB

Jumlah data kosong:
waktu                  0
lokasi                 0
suhu_celsius           0
kelembapan_persen      0
curah_hujan_mm         0
kecepatan_angin_kmh    0
tutupan_awan_persen    0
kode_cuaca             0
dtype: int64

Total record: 168
Dataset memenuhi minimal 100 recor

In [9]:
# Membuat lokasi penyimpanan
folder_data = "akuisisi-cuaca-bali/data/raw"

os.makedirs(folder_data, exist_ok=True)

# Menentukan nama file
file_csv = folder_data + "/data_cuaca_bali.csv"

# Menyimpan DataFrame
data.to_csv(file_csv, index=False)

print("Dataset berhasil disimpan!")
print("Lokasi file:", file_csv)

Dataset berhasil disimpan!
Lokasi file: akuisisi-cuaca-bali/data/raw/data_cuaca_bali.csv


In [10]:
# Mencatat waktu akses
waktu_akses = datetime.now(
    ZoneInfo("Asia/Singapore")
).isoformat()

# Menghitung hash MD5
with open(file_csv, "rb") as file:
    hash_md5 = hashlib.md5(file.read()).hexdigest()

# Membuat dokumentasi provenance
provenance = {
    "nama_dataset": "Data Cuaca Denpasar Bali",
    "sumber_data": "Open-Meteo",
    "url_api": response.url,
    "publisher": "Open-Meteo",
    "metode": "API menggunakan Python requests",
    "waktu_akses": waktu_akses,
    "lokasi": lokasi,
    "latitude": latitude,
    "longitude": longitude,
    "timezone": "Asia/Singapore",
    "tanggal_mulai": tanggal_mulai,
    "tanggal_selesai": tanggal_selesai,
    "jumlah_record": len(data),
    "jumlah_kolom": len(data.columns),
    "format_file": "CSV",
    "hash_md5": hash_md5
}

# Menyimpan provenance
folder_docs = "akuisisi-cuaca-bali/docs"
os.makedirs(folder_docs, exist_ok=True)

with open(
    folder_docs + "/provenance.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        provenance,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Provenance berhasil dibuat!")

Provenance berhasil dibuat!


In [11]:
checklist = pd.DataFrame([
    ["Sumber data publik", "Ya", "Open-Meteo"],
    ["Menggunakan API resmi", "Ya", "Historical Weather API"],
    ["Tidak mengambil data pribadi", "Ya", "Data cuaca"],
    ["Mencantumkan sumber data", "Ya", "Open-Meteo"],
    ["Menggunakan timeout", "Ya", "30 detik"],
    ["Memeriksa status respons", "Ya", "HTTP status"],
    ["Mencatat waktu akses", "Ya", "Asia/Singapore"],
    ["Mencatat hash dataset", "Ya", "MD5"]
], columns=[
    "pemeriksaan",
    "status",
    "keterangan"
])

file_checklist = (
    "akuisisi-cuaca-bali/docs/data_ethics_checklist.csv"
)

checklist.to_csv(file_checklist, index=False)

display(checklist)

print("Checklist etika berhasil dibuat!")

,pemeriksaan,status,keterangan
0,Sumber data publik,Ya,Open-Meteo
1,Menggunakan API resmi,Ya,Historical Weather API
2,Tidak mengambil data pribadi,Ya,Data cuaca
3,Mencantumkan sumber data,Ya,Open-Meteo
4,Menggunakan timeout,Ya,30 detik
5,Memeriksa status respons,Ya,HTTP status
6,Mencatat waktu akses,Ya,Asia/Singapore
7,Mencatat hash dataset,Ya,MD5


Checklist etika berhasil dibuat!
